<a href="https://colab.research.google.com/github/wikiwa1/ASD-with-SSL/blob/main/Baseline_model_with_logmel_spectrogram_avg_and_aucroc_metric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── imports ──────────────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import Audio, display

# ── global style ─────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 11

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT = Path('/content/drive/MyDrive/fan/')   # adjust if data lives elsewhere
MACHINE_IDS = ['id_00', 'id_02', 'id_04', 'id_06']
SR = 16_000               # native sample rate
CHANNEL = 0               # which mic channel to use (0–7); channel 0 is conventional

In [3]:
logmel_df = pd.read_pickle('/content/drive/MyDrive/fan/logmel_df.pkl')

In [4]:
logmel_df

,machine_id,label,is_anomaly,logmel
0,id_00,normal,0,"[[-17.983612, -12.184507, -13.16911, -15.99170..."
1,id_00,normal,0,"[[-25.626785, -19.677876, -16.160725, -18.2890..."
2,id_00,normal,0,"[[-16.12159, -16.24141, -11.531551, -11.166359..."
3,id_00,normal,0,"[[-19.059937, -15.340146, -15.898292, -15.3384..."
4,id_00,normal,0,"[[-14.151649, -9.280404, -9.742218, -13.508335..."
...,...,...,...,...
5545,id_06,abnormal,1,"[[-26.81831, -26.008778, -27.578476, -24.48549..."
5546,id_06,abnormal,1,"[[-19.232435, -19.220243, -18.999292, -24.0809..."
5547,id_06,abnormal,1,"[[-17.390596, -12.195385, -14.641667, -19.6757..."
5548,id_06,abnormal,1,"[[-13.803755, -15.295338, -21.245111, -18.6105..."


In [5]:
machine_dfs = {
    machine_id: logmel_df[logmel_df['machine_id'] == machine_id]
    for machine_id in logmel_df['machine_id'].unique()
}

df_id00 = machine_dfs['id_00']
df_id02 = machine_dfs['id_02']
df_id04 = machine_dfs['id_04']
df_id06 = machine_dfs['id_06']

In [6]:
df_id00

,machine_id,label,is_anomaly,logmel
0,id_00,normal,0,"[[-17.983612, -12.184507, -13.16911, -15.99170..."
1,id_00,normal,0,"[[-25.626785, -19.677876, -16.160725, -18.2890..."
2,id_00,normal,0,"[[-16.12159, -16.24141, -11.531551, -11.166359..."
3,id_00,normal,0,"[[-19.059937, -15.340146, -15.898292, -15.3384..."
4,id_00,normal,0,"[[-14.151649, -9.280404, -9.742218, -13.508335..."
...,...,...,...,...
1413,id_00,abnormal,1,"[[-19.000156, -18.773674, -16.105547, -13.0717..."
1414,id_00,abnormal,1,"[[-11.028221, -9.782515, -12.335272, -7.477394..."
1415,id_00,abnormal,1,"[[-16.773098, -16.093395, -18.591566, -17.3528..."
1416,id_00,abnormal,1,"[[-9.488977, -9.012589, -10.127343, -8.790489,..."


In [10]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    logmel_df,
    test_size=0.2,
    stratify=logmel_df['label'],
    random_state=42
)

In [9]:
from sklearn.model_selection import train_test_split

train_df_id00, test_df_id00 = train_test_split(
    df_id00,
    test_size=0.2,
    stratify=df_id00['label'],
    random_state=42
)

train_df_id02, test_df_id02 = train_test_split(
    df_id02,
    test_size=0.2,
    stratify=df_id02['label'],
    random_state=42
)

train_df_id04, test_df_id04 = train_test_split(
    df_id04,
    test_size=0.2,
    stratify=df_id04['label'],
    random_state=42
)

train_df_id06, test_df_id06 = train_test_split(
    df_id06,
    test_size=0.2,
    stratify=df_id06['label'],
    random_state=42
)

In [ ]:
import numpy as np

def pad_to_max(arrays):
    max_time = max(a.shape[1] for a in arrays)
    padded = [
        np.pad(a, ((0,0),(0, max_time - a.shape[1])),
               mode='constant', constant_values=a.min())
        for a in arrays
    ]
    return np.stack(padded)

averages = {}

for label, group in train_df.groupby('label'):
    stacked = pad_to_max(group['logmel'].tolist())
    avg_logmel = stacked.mean(axis=0)
    averages[label] = avg_logmel
    print(label, avg_logmel.shape)

avg_normal = averages['normal']
avg_abnormal = averages['abnormal']

abnormal (128, 313)
normal (128, 313)


In [11]:
import numpy as np

def pad_to_max(arrays):
    max_time = max(a.shape[1] for a in arrays)
    padded = [
        np.pad(a, ((0,0),(0, max_time - a.shape[1])),
               mode='constant', constant_values=a.min())
        for a in arrays
    ]
    return np.stack(padded)

averages = {}

for (machine_id, label), group in train_df.groupby(['machine_id', 'label']):
    stacked = pad_to_max(group['logmel'].tolist())
    avg_logmel = stacked.mean(axis=0)
    averages[(machine_id, label)] = avg_logmel
    print(machine_id, label, avg_logmel.shape)

id_00 abnormal (128, 313)
id_00 normal (128, 313)
id_02 abnormal (128, 313)
id_02 normal (128, 313)
id_04 abnormal (128, 313)
id_04 normal (128, 313)
id_06 abnormal (128, 313)
id_06 normal (128, 313)


In [12]:
print(averages)

{('id_00', 'abnormal'): array([[-17.394144 , -14.536211 , -14.545151 , ..., -14.160987 ,
        -14.340884 , -15.356972 ],
       [-13.394955 , -10.54172  , -10.683872 , ..., -10.507109 ,
        -10.799542 , -11.553926 ],
       [-11.207785 ,  -7.5271792,  -7.1534953, ...,  -6.806027 ,
         -7.1677423,  -8.575246 ],
       ...,
       [-54.32455  , -53.088665 , -53.531216 , ..., -53.509552 ,
        -53.47041  , -53.235992 ],
       [-55.19704  , -54.160385 , -54.75461  , ..., -54.71028  ,
        -54.66141  , -54.140347 ],
       [-56.695396 , -56.138706 , -57.01788  , ..., -57.02021  ,
        -56.796204 , -55.735176 ]], dtype=float32), ('id_00', 'normal'): array([[-17.87866 , -14.833067, -14.642044, ..., -14.568391, -14.689883,
        -15.893969],
       [-14.472758, -11.387465, -11.325297, ..., -11.471311, -11.428357,
        -12.353734],
       [-12.362974,  -8.678695,  -8.44626 , ...,  -8.409058,  -8.500781,
         -9.97317 ],
       ...,
       [-53.23427 , -51.807472, 

In [14]:
from sklearn.metrics import roc_auc_score
for machine_id in test_df['machine_id'].unique():
    machine_test = test_df[test_df['machine_id'] == machine_id]

    avg_normal = averages[(machine_id, 'normal')]

    true_labels = [1 if label == 'abnormal' else 0 for label in machine_test['label']]
    scores = [np.linalg.norm(sample - avg_normal) for sample in machine_test['logmel']]

    auc = roc_auc_score(true_labels, scores)
    print(f"{machine_id}: AUC = {auc:.4f}")

id_04: AUC = 0.4547
id_06: AUC = 0.6067
id_00: AUC = 0.5581
id_02: AUC = 0.6234


In [16]:
from sklearn.metrics import roc_auc_score
for machine_id in test_df['machine_id'].unique():
    machine_test = test_df[test_df['machine_id'] == machine_id]

    avg_abnormal = averages[(machine_id, 'abnormal')]

    true_labels = [1 if label == 'abnormal' else 0 for label in machine_test['label']]
    scores = [- np.linalg.norm(sample - avg_abnormal) for sample in machine_test['logmel']]

    auc = roc_auc_score(true_labels, scores)
    print(f"{machine_id}: AUC = {auc:.4f}")

id_04: AUC = 0.5849
id_06: AUC = 0.4863
id_00: AUC = 0.4790
id_02: AUC = 0.4002


In [17]:
from sklearn.metrics import roc_auc_score
for machine_id in test_df['machine_id'].unique():
    machine_test = test_df[test_df['machine_id'] == machine_id]

    avg_normal = averages[(machine_id, 'normal')]
    avg_abnormal = averages[(machine_id, 'abnormal')]

    true_labels = [1 if label == 'abnormal' else 0 for label in machine_test['label']]
    scores = [np.linalg.norm(sample - avg_normal) - np.linalg.norm(sample - avg_abnormal) for sample in machine_test['logmel']]

    auc = roc_auc_score(true_labels, scores)
    print(f"{machine_id}: AUC = {auc:.4f}")

id_04: AUC = 0.7553
id_06: AUC = 0.7598
id_00: AUC = 0.6087
id_02: AUC = 0.5326


Trying the cosine distance

In [20]:
from sklearn.metrics import roc_auc_score
from scipy.spatial.distance import cosine

for machine_id in test_df['machine_id'].unique():
    machine_test = test_df[test_df['machine_id'] == machine_id]

    avg_normal = averages[(machine_id, 'normal')]
    avg_abnormal = averages[(machine_id, 'abnormal')]

    true_labels = [1 if label == 'abnormal' else 0 for label in machine_test['label']]
    scores = [cosine(sample.flatten(), avg_normal.flatten())- cosine(sample.flatten(), avg_abnormal.flatten()) for sample in machine_test['logmel']]

    auc = roc_auc_score(true_labels, scores)
    print(f"{machine_id}: AUC = {auc:.4f}")

id_04: AUC = 0.7808
id_06: AUC = 0.8136
id_00: AUC = 0.6660
id_02: AUC = 0.9190
